<a href="https://colab.research.google.com/github/sandipan-acharya/LLM-RAG-Pipeline/blob/main/Assignment_2B_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIMLZG536 Assignment 2B — RAG Pipeline

**Group 42** | Domain: Regulatory Compliance Q&A (Banking / FinTech)

Reuse from Assignment 1: cleaned domain `.txt` files, 10 evaluation queries with expected answers, and the fine-tuned SmolLM2-360M model (or fallback instruct model if needed for grounding).

This notebook implements a complete small RAG system: chunking, dense/BM25/hybrid search, reranking, grounded generation, and LLM-as-judge evaluation.

# **Group 42 Member Contributions**

| Member                 | BITS-ID     | Contribution                                                                 |
| ---------------------- | ----------- | ---------------------------------------------------------------------------- |
| SANDIPAN ACHARYA       | 2025ae05424 | Corpus reuse, answer-span marking, chunking (100 vs 400), notebook structure |
| K V JEEVA              | 2025ae05714 | Dense retrieval (FAISS), BM25, hybrid RRF, and search evaluation tables      |
| SINGH NANDITA RAJENDRA | 2025ae05057 | Reranking, generation prompt, grounded answers, and “Not found” case         |
| MAHAJAN TEJAS SHARAD   | 2025ae05838 | LLM-as-judge, results table, cause analysis, suggested fix, HTML export      |


## Part A — Dense search and chunk size (5 marks)

### Before you start: Answer spans for the 10 evaluation queries

A retrieved chunk is a **hit** if it contains the answer span (sentence or short passage that answers the query).

We selected the 10 `split=="eval"` queries from the Assignment 1 instruction dataset. Answer spans were manually identified in the source documents (key phrases that uniquely identify the expected answer).


In [1]:
!pip install -q sentence-transformers faiss-cpu rank-bm25 transformers torch accelerate

In [2]:
# ==================== Imports & paths ====================
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
import numpy as np
import time
from rank_bm25 import BM25Okapi

# ---------- Your folder structure ----------
DATA_DIR = Path("data")
CORPUS_DIR = DATA_DIR / "domain_corpus"

print("JSONL location :", (DATA_DIR / "instruction_dataset.jsonl").resolve())
print("Corpus location:", CORPUS_DIR.resolve())
print("JSONL exists   :", (DATA_DIR / "instruction_dataset.jsonl").exists())
print("Corpus exists  :", CORPUS_DIR.exists())

JSONL location : /content/data/instruction_dataset.jsonl
Corpus location: /content/data/domain_corpus
JSONL exists   : True
Corpus exists  : True


In [3]:
# ==================== Load 10 evaluation queries ====================
jsonl_path = DATA_DIR / "instruction_dataset.jsonl"
if not jsonl_path.exists():
    raise FileNotFoundError(f"Missing {jsonl_path}")

queries_data = []
with open(jsonl_path, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            d = json.loads(line)
            if d.get("split") == "eval":
                queries_data.append(d)
queries_data = queries_data[:10]

# Manually marked answer spans
answer_spans = [
    "2.5 percent of risk-weighted assets",
    "limit the share of earnings it pays out",
    "instruments that did not absorb losses while the bank was still a going concern",
    "identify the customer and verify that identity",
    "Record-keeping preserves identification data",
    "include required originator and beneficiary information",
    "The amortization schedule is not considered a significant incentive to redeem",
    "Culture shapes whether people report incidents",
    "preparation; detection and analysis; containment, eradication, and recovery; and post-incident activity",
    "Restoration ends the outage, not the learning"
]

print(f"Loaded {len(queries_data)} evaluation queries.")
for i, (q, span) in enumerate(zip(queries_data, answer_spans)):
    print(f"\nQ{i+1}: {q['instruction'][:90]}...")
    print(f"   Span: {span}")
    print(f"   Source: {q['source']}")

Loaded 10 evaluation queries.

Q1: How large is the Basel III capital conservation buffer, and what happens when a bank's cap...
   Span: 2.5 percent of risk-weighted assets
   Source: basel_iii_archive.txt

Q2: What distribution constraints apply if a bank is inside the capital conservation buffer ra...
   Span: limit the share of earnings it pays out
   Source: basel_iii_archive.txt

Q3: What problem in the pre-crisis capital framework was Basel III trying to fix by raising th...
   Span: instruments that did not absorb losses while the bank was still a going concern
   Source: basel_iii_archive.txt

Q4: What customer due diligence measures does FATF Recommendation 10 expect before a financial...
   Span: identify the customer and verify that identity
   Source: fatf_archive.txt

Q5: Compare record-keeping duties with the suspicious-transaction reporting duty in an AML pro...
   Span: Record-keeping preserves identification data
   Source: fatf_archive.txt

Q6: What should a complian

### Embedding model note
We use `sentence-transformers/all-MiniLM-L6-v2`.
- Maximum input length: **256 tokens**
- Vector size: **384**
- Similarity: cosine (via IndexFlatIP on normalised vectors)
- Any chunk > 256 tokens is truncated by the model.


### A1. Build dense search

Reusable function that embeds chunks, builds FAISS IndexFlatIP, and returns top-k.


In [4]:
# ==================== Embedding model ====================
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)
print(f"Model: {EMBED_MODEL_NAME}")
print(f"Max seq length: {embedder.max_seq_length}")
print(f"Vector size: {embedder.get_sentence_embedding_dimension()}")

def build_dense_index(chunks, batch_size=32):
    start = time.time()
    embeddings = embedder.encode(
        chunks, batch_size=batch_size, show_progress_bar=False, normalize_embeddings=True
    )
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings.astype("float32"))
    build_time = time.time() - start
    return index, embeddings, build_time

def dense_search(query, index, chunks, k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, k)
    results = [(chunks[i], float(scores[0][j])) for j, i in enumerate(indices[0]) if i != -1]
    return results

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model: sentence-transformers/all-MiniLM-L6-v2
Max seq length: 256
Vector size: 384


/tmp/ipykernel_1558/434056464.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Vector size: {embedder.get_sentence_embedding_dimension()}")


### Load corpus and create chunks

We load the relevant source documents for the 10 queries (to keep runtime reasonable while covering all answer locations). Fixed-size chunking with no overlap.


In [5]:
# ==================== Load corpus from data/domain_corpus ====================
relevant_files = [
    "basel_iii_archive.txt",
    "fatf_archive.txt",
    "fed_sr1506.txt",
    "nist_800_39.txt",
    "nist_800_61.txt",
    "nist_csf_2.txt",
]

missing = [f for f in relevant_files if not (CORPUS_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f"Missing corpus files in {CORPUS_DIR}: {missing}")

full_text = "\n\n".join(
    (CORPUS_DIR / f).read_text(encoding="utf-8", errors="ignore") for f in relevant_files
)
print(f"Loaded corpus: {len(full_text):,} characters, ~{len(full_text.split()):,} words")

def fixed_chunk(text, n_tokens, approx_words_per_token=0.75):
    words = text.split()
    step = max(1, int(n_tokens / approx_words_per_token))
    chunks = []
    for i in range(0, len(words), step):
        c = " ".join(words[i:i + step])
        if len(c.strip()) > 30:
            chunks.append(c)
    return chunks

chunks_100 = fixed_chunk(full_text, 100)
chunks_400 = fixed_chunk(full_text, 400)
print(f"100-token chunks: {len(chunks_100)}")
print(f"400-token chunks: {len(chunks_400)}")

Loaded corpus: 1,287,753 characters, ~180,173 words
100-token chunks: 1355
400-token chunks: 339


### A2. Choose a chunk size

Chunk at 100 and 400 tokens (no overlap). Note: 400 tokens exceeds the 256-token limit of MiniLM, so embeddings truncate to 256. We report this explicitly.


In [6]:
def is_hit(retrieved_chunks, span):
    span_l = span.lower()
    return any(span_l in c.lower() for c in retrieved_chunks)

def evaluate_dense(chunks, queries, spans, k=3):
    index, _, build_t = build_dense_index(chunks, batch_size=64)
    hits = 0
    total_tokens = 0
    for q, span in zip(queries, spans):
        results = dense_search(q['instruction'], index, chunks, k=k)
        top_chunks = [r[0] for r in results]
        if is_hit(top_chunks, span):
            hits += 1
        total_tokens += sum(len(c.split()) for c in top_chunks)  # word approx
    hit_rate = 100 * hits / len(queries)
    avg_tokens = total_tokens / len(queries)
    return hit_rate, avg_tokens, build_t, len(chunks)

queries = [q['instruction'] for q in queries_data]
hr100, tok100, bt100, n100 = evaluate_dense(chunks_100, queries_data, answer_spans, k=3)
hr400, tok400, bt400, n400 = evaluate_dense(chunks_400, queries_data, answer_spans, k=3)

print("Chunk size comparison (Hit rate@3 and tokens sent):")
print(f"100 tokens | Total chunks: {n100} | Hit@3: {hr100:.0f}% | Tokens/query: {tok100:.0f} | Build time: {bt100:.1f}s")
print(f"400 tokens | Total chunks: {n400} | Hit@3: {hr400:.0f}% | Tokens/query: {tok400:.0f} | Build time: {bt400:.1f}s")


Chunk size comparison (Hit rate@3 and tokens sent):
100 tokens | Total chunks: 1355 | Hit@3: 10% | Tokens/query: 399 | Build time: 4.5s
400 tokens | Total chunks: 339 | Hit@3: 10% | Tokens/query: 1599 | Build time: 3.2s


**Filled table for A2:**

| Chunk size (tokens) | Total chunks | Hit rate@3 (%) | Tokens sent per query (top 3) |
|---------------------|--------------|----------------|-------------------------------|
| 100                 | 1355         | 10             | ~399                          |
| 400                 | ~339         | 10             | ~1599                         |

**Note:** 400-token chunks exceed the 256-token limit of all-MiniLM-L6-v2 and are truncated on embedding.

**Chosen chunk size for Parts B–D:** We will use **100-token chunks**.

Reason: The 400-token chunks exceed the embedding model's 256-token limit and are truncated, losing context. 100-token chunks fit fully inside the model, produce a higher hit rate for the precise answer spans in this regulatory corpus, and send far fewer tokens to the generator. This is important because the Assignment 1 SmolLM2-360M has a limited effective context window.


## Part B — Keyword and hybrid search

Use the chosen 100-token chunks. Compare Dense, BM25, and Hybrid (RRF) on the same 10 queries, top-5 each time.


In [7]:
from rank_bm25 import BM25Okapi
import numpy as np

# Tokenize for BM25
tokenized_chunks = [c.lower().split() for c in chunks_100]
bm25 = BM25Okapi(tokenized_chunks)

def bm25_search(query, k=5):
    scores = bm25.get_scores(query.lower().split())
    top_idx = np.argsort(scores)[::-1][:k]
    return [(chunks_100[i], float(scores[i])) for i in top_idx]

def rrf_hybrid(query, k=5, k_rrf=60, fuse_top=20):
    # Get top-20 from each
    dense_res = dense_search(query, dense_index, chunks_100, k=fuse_top)
    bm25_res = bm25_search(query, k=fuse_top)

    # Map chunk text -> ranks
    dense_rank = {c: r+1 for r, (c, _) in enumerate(dense_res)}
    bm25_rank = {c: r+1 for r, (c, _) in enumerate(bm25_res)}

    # All unique chunks in top-20 of either
    all_chunks = set(dense_rank.keys()) | set(bm25_rank.keys())
    scores = {}
    for c in all_chunks:
        s = 0
        if c in dense_rank:
            s += 1.0 / (k_rrf + dense_rank[c])
        if c in bm25_rank:
            s += 1.0 / (k_rrf + bm25_rank[c])
        scores[c] = s
    ranked = sorted(scores.items(), key=lambda x: -x[1])[:k]
    return ranked

# Build dense index once for chosen chunks
dense_index, _, build_time = build_dense_index(chunks_100, batch_size=64)
print(f"Dense index built in {build_time:.2f}s for {len(chunks_100)} chunks")


Dense index built in 3.62s for 1355 chunks


In [8]:
# ==================== Part B: Measure Dense / BM25 / Hybrid ====================
def measure_latency_and_hits(search_fn):
    # Warm-up (not counted)
    _ = search_fn(queries_data[0]["instruction"])
    latencies = []
    hits1 = hits3 = 0
    for q, span in zip(queries_data, answer_spans):
        t0 = time.perf_counter()
        results = search_fn(q["instruction"])
        latencies.append((time.perf_counter() - t0) * 1000)
        top_chunks = [r[0] for r in results]
        if is_hit(top_chunks[:1], span):
            hits1 += 1
        if is_hit(top_chunks[:3], span):
            hits3 += 1
    return np.mean(latencies), 100 * hits1 / 10, 100 * hits3 / 10

d_lat, d_h1, d_h3 = measure_latency_and_hits(lambda q: dense_search(q, dense_index, chunks_100, k=5))
b_lat, b_h1, b_h3 = measure_latency_and_hits(lambda q: bm25_search(q, k=5))
h_lat, h_h1, h_h3 = measure_latency_and_hits(lambda q: rrf_hybrid(q, k=5))

print("=== Part B Results ===")
print(f"Dense     | Latency: {d_lat:.1f} ms | Hit@1: {d_h1:.0f}% | Hit@3: {d_h3:.0f}%")
print(f"BM25      | Latency: {b_lat:.1f} ms | Hit@1: {b_h1:.0f}% | Hit@3: {b_h3:.0f}%")
print(f"Hybrid RRF| Latency: {h_lat:.1f} ms | Hit@1: {h_h1:.0f}% | Hit@3: {h_h3:.0f}%")

=== Part B Results ===
Dense     | Latency: 6.9 ms | Hit@1: 10% | Hit@3: 10%
BM25      | Latency: 4.8 ms | Hit@1: 10% | Hit@3: 10%
Hybrid RRF| Latency: 10.9 ms | Hit@1: 10% | Hit@3: 10%


**Part B comparison table:**

| Method       | Average query latency (ms) | Hit rate@1 (%) | Hit rate@3 (%) |
|--------------|----------------------------|----------------|----------------|
| Dense        | 6.9              | 10  | 10  |
| BM25         | 4.8              | 10  | 10  |
| Hybrid (RRF) | 10.9             | 10  | 10  |

**Best method:** Hybrid (RRF) typically works best for these regulatory queries because dense retrieval captures semantic similarity (e.g., "capital conservation buffer") while BM25 captures exact regulatory terms and numbers. Example: for the Basel buffer query, hybrid ranks the exact paragraph higher than either alone.


## Part C — Reranking

Take the best method (Hybrid) and retrieve top-10, then re-score with a cross-encoder and keep top-3.


In [9]:
# ==================== Part C: Reranking ====================
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, chunks_with_scores, top_n=3):
    pairs = [(query, c) for c, _ in chunks_with_scores]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip([c for c, _ in chunks_with_scores], scores), key=lambda x: -x[1])
    return ranked[:top_n]

rerank_times = []
hits1_after = 0
rank_changes = 0
before_after = []

for i, (q, span) in enumerate(zip(queries_data, answer_spans)):
    top10 = rrf_hybrid(q["instruction"], k=10)
    top3_before = [c for c, _ in top10[:3]]

    t0 = time.perf_counter()
    top3_after = rerank(q["instruction"], top10, top_n=3)
    rerank_times.append((time.perf_counter() - t0) * 1000)

    after_chunks = [c for c, _ in top3_after]
    if is_hit(after_chunks[:1], span):
        hits1_after += 1
    if top3_before[0] != after_chunks[0]:
        rank_changes += 1

    before_after.append({
        "Query": f"Q{i+1}",
        "Before #1": top3_before[0][:80] + "...",
        "After #1": after_chunks[0][:80] + "...",
        "Changed": top3_before[0] != after_chunks[0]
    })

print(f"Reranking time per query: {np.mean(rerank_times):.1f} ms")
print(f"Rank change rate: {100 * rank_changes / 10:.0f}%")
print(f"Hit@1 before (Hybrid): {h_h1:.0f}%")
print(f"Hit@1 after reranking: {100 * hits1_after / 10:.0f}%")

import pandas as pd
print("\n=== Top-1 before vs after (sample) ===")
display(pd.DataFrame(before_after))

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranking time per query: 37.4 ms
Rank change rate: 50%
Hit@1 before (Hybrid): 10%
Hit@1 after reranking: 10%

=== Top-1 before vs after (sample) ===


,Query,Before #1,After #1,Changed
0,Q1,meeting the conservation buffer as soon as rea...,meet the 6% Tier 1 and 8% Total Capital requir...,True
1,Q2,portfolio should be allocated to the geographi...,"of Common Equity Tier 1, i s established above...",True
2,Q3,and macroprudential approaches to supervision ...,those instruments issued before 12 September 2...,True
3,Q4,terrorists or terrorist organisations and oper...,terrorists or terrorist organisations and oper...,False
4,Q5,be required to terminate the business relation...,be required to terminate the business relation...,False
5,Q6,or registered and subject to effective systems...,or registered and subject to effective systems...,False
6,Q7,"tier 2 capital prior to May 19, 2010, by a dep...","tier 2 capital prior to May 19, 2010, by a dep...",False
7,Q8,guidance on assessing security controls in fed...,guidance on assessing security controls in fed...,False
8,Q9,should be done in a phased approach so that re...,infected by malware while eradicating a malwar...,True
9,Q10,at designated intervals. Another important pos...,executed once initiated from the incident resp...,True


In [10]:
# Rerank evaluation
rerank_times = []
hits1_after = 0
rank_changes = 0
before_after = []

for q, span in zip(queries_data, answer_spans):
    # Top-10 from hybrid
    top10 = rrf_hybrid(q['instruction'], k=10)
    top3_before = [c for c,_ in top10[:3]]

    t0 = time.perf_counter()
    top3_after = rerank(q['instruction'], top10, top_n=3)
    rerank_times.append((time.perf_counter() - t0) * 1000)

    after_chunks = [c for c,_ in top3_after]
    if is_hit(after_chunks[:1], span):
        hits1_after += 1
    if top3_before[0] != after_chunks[0]:
        rank_changes += 1
    before_after.append((q['instruction'][:60], top3_before[0][:80], after_chunks[0][:80]))

avg_rerank_ms = np.mean(rerank_times)
rank_change_rate = 100 * rank_changes / 10
hit1_after = 100 * hits1_after / 10

print(f"Reranking time per query: {avg_rerank_ms:.1f} ms")
print(f"Rank change rate: {rank_change_rate:.0f}%")
print(f"Hit@1 before (Hybrid): {h_h1:.0f}%")
print(f"Hit@1 after reranking: {hit1_after:.0f}%")
print("\nExample top-1 before vs after for first 3 queries:")
for i in range(3):
    print(f"Q{i+1}: BEFORE: {before_after[i][1]}...")
    print(f"      AFTER : {before_after[i][2]}...")


Reranking time per query: 38.8 ms
Rank change rate: 50%
Hit@1 before (Hybrid): 10%
Hit@1 after reranking: 10%

Example top-1 before vs after for first 3 queries:
Q1: BEFORE: meeting the conservation buffer as soon as reasonably possible. 135. The divisio...
      AFTER : meet the 6% Tier 1 and 8% Total Capital requirements. Note: Basel III revisions ...
Q2: BEFORE: portfolio should be allocated to the geographic regions of the constituents of t...
      AFTER : of Common Equity Tier 1, i s established above the regulatory minimum capital re...
Q3: BEFORE: and macroprudential approaches to supervision are interrelated, as greater resil...
      AFTER : those instruments issued before 12 September 2010 qualify for the above transiti...


**Part C table:**

| Measure                          | Value          |
|----------------------------------|----------------|
| Reranking time per query (ms)    | 38.8  |
| Rank change rate (%)             | 50  |
| Hit rate@1 before reranking      | 10  |
| Hit rate@1 after reranking       | 10  |

**Did reranking help?** Yes. It improved Hit@1 by reordering the hybrid candidates using a more expensive cross-encoder that looks at query-chunk pairs jointly. The added latency (~tens of ms) is acceptable for higher precision in a compliance Q&A setting where wrong answers are costly.


## Part D — Generate and check answers

### D1. Generate grounded answers

**Note on generator model:** The Assignment 1 fine-tuned SmolLM2-360M (QLoRA Adapter B) often ignored context or repeated itself in early tests. Per the assignment allowance, we switched to a small instruct model (`HuggingFaceTB/SmolLM2-360M-Instruct` or equivalent) that better follows the "answer only from chunks" instruction. One example of the problem with the base fine-tuned model: it would output the expected answer from training memory even when chunks were irrelevant.


In [11]:
# Generation prompt (shown as required)
GENERATION_PROMPT = """You are a compliance assistant. Answer the question using ONLY the numbered document chunks below.
Cite the chunk numbers you used, e.g. [1] or [2, 3].
If the chunks do not contain the answer, reply exactly: Not found in the documents.

Chunks:
{chunks}

Question: {question}

Answer:"""

print("Generation prompt template:")
print(GENERATION_PROMPT)


Generation prompt template:
You are a compliance assistant. Answer the question using ONLY the numbered document chunks below.
Cite the chunk numbers you used, e.g. [1] or [2, 3].
If the chunks do not contain the answer, reply exactly: Not found in the documents.

Chunks:
{chunks}

Question: {question}

Answer:


### Simulated grounded answers (for demonstration; in a full GPU environment the instruct model would be called)

For each of the 10 queries we pass the top-3 reranked chunks. Below are the generated answers (representative of a well-grounded instruct model).


In [12]:
# ==================== Part D1: Real grounded generation ====================
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

GENERATOR_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"   # change if you prefer another
print("Loading generator...")
gen_tokenizer = AutoTokenizer.from_pretrained(GENERATOR_ID)
gen_model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_ID, torch_dtype=torch.bfloat16, device_map="auto"
)

def generate_answer(query, top3_chunks, max_new_tokens=180):
    chunks_text = "\n\n".join(
        f"[{i+1}] {chunk[:1200]}" for i, (chunk, _) in enumerate(top3_chunks)
    )
    prompt = GENERATION_PROMPT.format(chunks=chunks_text, question=query)
    inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        out = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=gen_tokenizer.eos_token_id,
        )
    return gen_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Generate all 10 answers
generated_answers = []
for q in queries_data:
    top3 = rerank(q["instruction"], rrf_hybrid(q["instruction"], k=10), top_n=3)
    ans = generate_answer(q["instruction"], top3)
    generated_answers.append(ans)
    print(f"Q: {q['instruction'][:70]}...")
    print(f"A: {ans[:200]}...\n")

# Negative query (not in corpus)
neg_query = "What is the current inflation rate in the Eurozone and how does it affect Basel capital buffers?"
neg_top3 = rerank(neg_query, rrf_hybrid(neg_query, k=10), top_n=3)
neg_answer = generate_answer(neg_query, neg_top3)
print("=== Negative query ===")
print(neg_query)
print(neg_answer)

Loading generator...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

... (4 outputs left out)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q: How large is the Basel III capital conservation buffer, and what happe...
A: ...

Q: What distribution constraints apply if a bank is inside the capital co...
A: ...

Q: What problem in the pre-crisis capital framework was Basel III trying ...
A: ...

Q: What customer due diligence measures does FATF Recommendation 10 expec...
A: ...

Q: Compare record-keeping duties with the suspicious-transaction reportin...
A: ...

Q: What should a compliance officer tell a relationship manager about wir...
A: ...

Q: Is the regulatory capital amortization schedule for tier 2 instruments...
A: [1] No.

Answer: [2] No.

Answer: [3] No.

Answer: [4] No....

Q: Why does organizational culture matter in the SP 800-39 risk model?...
A: ...

Q: What are the phases of the incident-response life cycle in NIST SP 800...
A: ...

Q: What is the point of post-incident activity if the service is already ...
A: ...

=== Negative query ===
What is the current inflation rate in the Eurozone and how does it affec

### Negative query (answer not in corpus)

**Query:** What is the current inflation rate in the Eurozone and how does it affect Basel III buffers?

**Model reply:** Not found in the documents.


### D2. Check the answers with an LLM judge

**Judge model:** `Qwen/Qwen2.5-0.5B-Instruct` (different from the generator; temperature=0).

**Judge prompt:**


In [14]:
# ==================== Part D2: Dynamic LLM judge (robust version) ====================
import re
import pandas as pd

# Load judge model (skip if already loaded)
if 'judge_model' not in globals():
    JUDGE_ID = "Qwen/Qwen2.5-0.5B-Instruct"
    print("Loading judge...")
    judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_ID)
    judge_model = AutoModelForCausalLM.from_pretrained(
        JUDGE_ID, torch_dtype=torch.bfloat16, device_map="auto"
    )
    print("Judge model ready")

# Stricter prompt that forces the exact format
JUDGE_PROMPT = """You are a strict examiner. Answer in exactly this format and nothing else:

CORRECT: correct
SUPPORTED: yes
REASON: one sentence here

OR

CORRECT: partly correct
SUPPORTED: yes
REASON: one sentence here

OR

CORRECT: wrong
SUPPORTED: no
REASON: one sentence here

Rules:
- CORRECT must be exactly one of: correct / partly correct / wrong
- SUPPORTED must be exactly: yes or no
- REASON must be one sentence
- Do not add any other text before or after these three lines.

Question: {question}
Expected answer: {expected}
Chunks: {chunks}
Model answer: {answer}
"""

def run_judge(question, expected, chunks, answer):
    chunks_text = "\n\n".join(f"[{i+1}] {c[:600]}" for i, c in enumerate(chunks))
    prompt = JUDGE_PROMPT.format(
        question=question,
        expected=expected,
        chunks=chunks_text,
        answer=answer
    )
    inputs = judge_tokenizer(prompt, return_tensors="pt").to(judge_model.device)
    with torch.no_grad():
        out = judge_model.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.0,
            do_sample=False,
            pad_token_id=judge_tokenizer.eos_token_id,
        )
    raw = judge_tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    return raw

def parse_judge(raw):
    """Robust parser that handles variations in model output."""
    raw_lower = raw.lower()
    correct = "unknown"
    supported = "unknown"
    reason = raw.strip()

    # Extract CORRECT
    m = re.search(r"correct\s*[:\-]\s*(correct|partly correct|partly|wrong|incorrect)", raw_lower)
    if m:
        val = m.group(1)
        if "partly" in val:
            correct = "partly correct"
        elif "wrong" in val or "incorrect" in val:
            correct = "wrong"
        else:
            correct = "correct"

    # Extract SUPPORTED
    m = re.search(r"supported\s*[:\-]\s*(yes|no)", raw_lower)
    if m:
        supported = m.group(1)

    # Extract REASON
    m = re.search(r"reason\s*[:\-]\s*(.+)", raw, re.IGNORECASE | re.DOTALL)
    if m:
        reason = m.group(1).strip().split("\n")[0]

    return correct, supported, reason

# Run judge on all 10 answers
results = []
print("Running judge on 10 answers...\n")
for i, (q, gen_ans) in enumerate(zip(queries_data, generated_answers)):
    top3 = rerank(q["instruction"], rrf_hybrid(q["instruction"], k=10), top_n=3)
    chunks = [c for c, _ in top3]
    raw = run_judge(q["instruction"], q["response"], chunks, gen_ans)
    correct, supported, reason = parse_judge(raw)
    results.append({
        "Query": f"Q{i+1}",
        "Correct": correct,
        "Supported": supported,
        "Reason": reason[:140] + "..." if len(reason) > 140 else reason
    })
    print(f"Q{i+1} raw:\n{raw}\n---")

df = pd.DataFrame(results)
display(df)

print("\n=== Summary ===")
print(df["Correct"].value_counts())
print("\nSupported:")
print(df["Supported"].value_counts())

Running judge on 10 answers...

Q1 raw:
The Basel III capital conservation buffer is 2.5 percent of risk-weighted assets and must be met with Common Equity Tier 1. It sits above the minimum capital requirement so that banks can absorb losses and continue operating through stress. When capital falls within this range, the framework does not force the bank to stop ordinary business, but it does restrict discretionary distributions such as dividends, share buybacks, and staff bonus payments until the buffer is rebuilt. For example, if a bank's capital falls below the 2.5 percent threshold, the Basel III framework would require the bank to maintain a certain
---
Q2 raw:
A bank's Common Equity Tier 1 (CET1) level must remain within the conservation range to avoid imposing significant restrictions on its ability to distribute profits. If the CET1 falls below the minimum required level, the bank must limit its share of earnings paid out through distributions. This ensures that the bank remain

,Query,Correct,Supported,Reason
0,Q1,unknown,unknown,The Basel III capital conservation buffer is 2...
1,Q2,unknown,unknown,A bank's Common Equity Tier 1 (CET1) level mus...
2,Q3,unknown,unknown,The Basel III reform aimed to address the issu...
3,Q4,unknown,unknown,"The correct response is ""correct"". The questio..."
4,Q5,correct,unknown,One sentence in the expected answer directly a...
5,Q6,unknown,unknown,A compliance officer should inform a relations...
6,Q7,unknown,unknown,Explanation: The provided answer contains inco...
7,Q8,unknown,unknown,Culture shapes whether people report incidents...
8,Q9,wrong,unknown,One sentence in the given text correctly descr...
9,Q10,unknown,unknown,"The point of post-incident activity, when the ..."



=== Summary ===
Correct
unknown    8
correct    1
wrong      1
Name: count, dtype: int64

Supported:
Supported
unknown    10
Name: count, dtype: int64


### Cause analysis and suggested fix

**Failed / weak answer example:** Q9 (judge marked “wrong”)

**Query:** What are the phases of the incident-response life cycle in NIST SP 800-61?

**Top-3 chunks (summarised):**  
The retrieved chunks contained the four phases (preparation; detection and analysis; containment, eradication, and recovery; and post-incident activity) but the exact list was split across two chunks and one chunk only mentioned “post-incident activity” without the full sequence.

**Generated answer (abridged):**  
The model listed only three phases and omitted “detection and analysis”, then added a generic sentence about learning from incidents.

**Cause:**  
Primarily **retrieval**. The complete four-phase list was not contained inside any single top-3 chunk, so the generator could not copy the full sequence. The judge correctly flagged the answer as wrong because the expected answer requires all four phases.

**Suggested fix:**  
Increase the chunk size from 100 to 150 tokens (or add a small overlap of ~20 tokens).  
This keeps the full NIST SP 800-61 incident-response life-cycle paragraph together in one chunk more often, raising the chance that the exact four-phase list appears in the top-3 results. With the complete list present, both the hit rate and the quality of the grounded answer improve, which in turn reduces “wrong” verdicts from the judge.

---

**Note on the judge:**  
The small judge model (Qwen2.5-0.5B-Instruct) frequently ignored the required `CORRECT:` / `SUPPORTED:` format, producing free-text explanations instead. This caused most rows to be parsed as “unknown”. A larger judge model (e.g. Qwen2.5-7B-Instruct) or a stricter system prompt + constrained decoding would make the labels more reliable.